# Methodology Overview

Comprehensive documentation of the ODE-Multiomics Benchmark experimental design.

In [1]:
import sys
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
from src.reynolds_ode import REYNOLDS_PARAMS

## Reynolds ODE Model Parameters

The benchmark is based on the Reynolds et al. (2006) acute inflammation model.

![Reynolds ODE System Diagram](../figures/schematics/Fig%206%20-%20ChatGPT%20Image%20Jul%2021%202026%2012_21_16%20PM%20-%20best.png)

![Reynolds ODE Bistability & Inflammation Outcomes](../figures/schematics/Fig%2010%20-%20ChatGPT%20Image%20Jul%2021%202026%2012_40_04%20PM%20-%20best.png)

In [2]:
# Display parameters as a structured table
param_categories = {
    'Pathogen Dynamics': ['k_pg', 'P_inf', 'k_pm', 's_dm', 'mu_p'],
    'Pro-inflammatory (N*)': ['s_nr', 'epsilon_nr', 's_nm', 's_nd', 'N_inf', 'mu_nr'],
    'DAMP (D)': ['k_dn', 'k_df', 'mu_d'],
    'Anti-inflammatory (CA)': ['s_c', 's_cn', 'mu_c'],
    'Tissue Damage': ['k_f', 'k_fh']
}
for category, params in param_categories.items():
    print(f'\n{category}:')
    print('-' * 50)
    for param in params:
        value = REYNOLDS_PARAMS[param]
        print(f'  {param:15s} = {value:10.4f}')


Pathogen Dynamics:
--------------------------------------------------
  k_pg            =     0.5000
  P_inf           =    20.0000
  k_pm            =     1.8000
  s_dm            =     0.0500
  mu_p            =     0.2000

Pro-inflammatory (N*):
--------------------------------------------------
  s_nr            =     0.0800
  epsilon_nr      =     0.3200
  s_nm            =     0.1000
  s_nd            =     0.1000
  N_inf           =     0.6000
  mu_nr           =     0.1200

DAMP (D):
--------------------------------------------------
  k_dn            =     0.0200
  k_df            =     0.0200
  mu_d            =     0.0500

Anti-inflammatory (CA):
--------------------------------------------------
  s_c             =     0.0400
  s_cn            =     0.0800
  mu_c            =     0.1000

Tissue Damage:
--------------------------------------------------
  k_f             =     0.0100
  k_fh            =     0.1200


![Synthetic Cohort Generation: Parameter Distributions & Outcome Trajectories](../figures/schematics/Fig%207%20-%20Gemini_Generated_Image_tkpo9ctkpo9ctkpo%20-%20best.png)

## Synthetic Data Generation

How synthetic patients are created with controlled heterogeneity.

In [3]:
print('Synthetic Data Generation Strategy:')
print('=' * 60)
print()
print('1. PARAMETER NOISE')
print('   - Lognormal distribution: sigma = 0.15 (plus/minus 15% log-scale)')
print('   - Per-patient parameters: p_i = p_base * exp(N(0, 0.15))')
print('   - Reflects natural inter-patient variability')
print()
print('2. MEASUREMENT NOISE')
print('   - Gaussian proportional: sigma = 0.10 (10% coefficient of variation)')
print('   - Observation model: y_obs = y_true * (1 + N(0, 0.10))')
print('   - Realistic sensor/assay noise')
print()
print('3. OBSERVATION TIMEPOINTS')
timepoints = np.array([0, 6, 12, 24, 48, 72])
print(f'   - Timepoints (hours): {timepoints}')
print(f'   - Number of measurements: {len(timepoints)}')
print(f'   - Clinical schedule: 0, 6h, 12h, 1d, 2d, 3d')
print()
print('4. OUTCOME STRATIFICATION')
print('   - Resolution:  low P0  (0.5 - 3.5)  -> antibodies clear pathogen')
print('   - Chronic:     mid P0  (4.0 - 9.0)  -> persistent inflammation')
print('   - Death:      high P0 (11.0-20.0)  -> runaway inflammation')

Synthetic Data Generation Strategy:

1. PARAMETER NOISE
   - Lognormal distribution: sigma = 0.15 (plus/minus 15% log-scale)
   - Per-patient parameters: p_i = p_base * exp(N(0, 0.15))
   - Reflects natural inter-patient variability

2. MEASUREMENT NOISE
   - Gaussian proportional: sigma = 0.10 (10% coefficient of variation)
   - Observation model: y_obs = y_true * (1 + N(0, 0.10))
   - Realistic sensor/assay noise

3. OBSERVATION TIMEPOINTS
   - Timepoints (hours): [ 0  6 12 24 48 72]
   - Number of measurements: 6
   - Clinical schedule: 0, 6h, 12h, 1d, 2d, 3d

4. OUTCOME STRATIFICATION
   - Resolution:  low P0  (0.5 - 3.5)  -> antibodies clear pathogen
   - Chronic:     mid P0  (4.0 - 9.0)  -> persistent inflammation
   - Death:      high P0 (11.0-20.0)  -> runaway inflammation


## MOTIF PipelineODE-based Multi-Omics Temporal Integration Framework

![MOTIF vs UDE: Pros and Cons Comparison](../figures/schematics/Fig%205%20-%20Gemini_Generated_Image_n51y1tn51y1tn51y%20-%20best.png)

In [4]:
print('MOTIF Pipeline Overview:')
print('=' * 60)
print()
print('STEP 1: ODE CALIBRATION')
print('  - Optimize parameters: k_pg, mu_p, s_nr, s_c, P0')
print('  - Method: L-BFGS-B optimization')
print('  - Objective: Minimize squared residuals on observed (N*, CA, f)')
print('  - Restarts: 5+1 to escape local minima')
print()
print('STEP 2: PROXY GENERATION')
print('  - Simulate calibrated ODE with fitted parameters')
print('  - Extract latent trajectories: P(t), D(t), h(t)')
print('  - Compute summary statistics: AUC, peak, final values')
print()
print('STEP 3: FEATURE EXTRACTION')
print('  - Observed features: AUC, peak, final for N*, CA, f')
print('  - Proxy features: AUC, peak, final for P, D, h')
print('  - Total features: ~18 dimensions')
print()
print('STEP 4: CORRELATION ANALYSIS')
print('  - Spearman rank correlation: features vs outcome')
print('  - Identify strongest biomarkers')
print()
print('STEP 5: CLASSIFICATION')
print('  - Logistic regression: features -> outcome')
print('  - Evaluate: AUROC, F1 score')
print('  - Compare: observed features vs with proxies')

MOTIF Pipeline Overview:

STEP 1: ODE CALIBRATION
  - Optimize parameters: k_pg, mu_p, s_nr, s_c, P0
  - Method: L-BFGS-B optimization
  - Objective: Minimize squared residuals on observed (N*, CA, f)
  - Restarts: 5+1 to escape local minima

STEP 2: PROXY GENERATION
  - Simulate calibrated ODE with fitted parameters
  - Extract latent trajectories: P(t), D(t), h(t)
  - Compute summary statistics: AUC, peak, final values

STEP 3: FEATURE EXTRACTION
  - Observed features: AUC, peak, final for N*, CA, f
  - Proxy features: AUC, peak, final for P, D, h
  - Total features: ~18 dimensions

STEP 4: CORRELATION ANALYSIS
  - Spearman rank correlation: features vs outcome
  - Identify strongest biomarkers

STEP 5: CLASSIFICATION
  - Logistic regression: features -> outcome
  - Evaluate: AUROC, F1 score
  - Compare: observed features vs with proxies


## UDE Pipeline

Universal Differential 

Equations + Symbolic Regression

In [5]:
print('UDE Pipeline Overview:')
print('=' * 60)
print()
print('STEP 1: NEURAL NETWORK ARCHITECTURE')
print('  - Input: Observed state [N*, CA, f]')
print('  - Hidden layers: 2 x tanh')
print('  - Output: Scalar clearance rate correction')
print('  - Non-linearity: Softplus (ensures non-negative rates)')
print()
print('STEP 2: HYBRID ODE SYSTEM')
print('  - Known dynamics: 4 of 5 equations from Reynolds model')
print('  - Learned dynamics: Pathogen clearance theta(N*, CA, f)')
print('  - State: [P, N*, D, CA, f]')
print('  - Training integrator: torchdiffeq with RK4')
print()
print('STEP 3: TRAINING')
print('  - Loss: MSE on observed variables')
print('  - Optimizer: Adam')
print('  - LR schedule: ReduceLROnPlateau')
print('  - Typical: 50-500 epochs, batch size 4-16')
print()
print('STEP 4: SYMBOLIC REGRESSION (OPTIONAL)')
print('  - Method: SINDy (Sparse Identification of Nonlinear Dynamics)')
print('  - Extract: Interpretable equations from learned function')
print('  - e.g., theta(N*) = c0 + c1*N* + c2*N*^2 + ...')

UDE Pipeline Overview:

STEP 1: NEURAL NETWORK ARCHITECTURE
  - Input: Observed state [N*, CA, f]
  - Hidden layers: 2 x tanh
  - Output: Scalar clearance rate correction
  - Non-linearity: Softplus (ensures non-negative rates)

STEP 2: HYBRID ODE SYSTEM
  - Known dynamics: 4 of 5 equations from Reynolds model
  - Learned dynamics: Pathogen clearance theta(N*, CA, f)
  - State: [P, N*, D, CA, f]
  - Training integrator: torchdiffeq with RK4

STEP 3: TRAINING
  - Loss: MSE on observed variables
  - Optimizer: Adam
  - LR schedule: ReduceLROnPlateau
  - Typical: 50-500 epochs, batch size 4-16

STEP 4: SYMBOLIC REGRESSION (OPTIONAL)
  - Method: SINDy (Sparse Identification of Nonlinear Dynamics)
  - Extract: Interpretable equations from learned function
  - e.g., theta(N*) = c0 + c1*N* + c2*N*^2 + ...


## Experimental Conditions

Six benchmark experiments testing different robustness scenarios.

In [6]:
print('Benchmark Experiments:')
print('=' * 70)
print()
experiments = [
    {
        'name': 'Baseline',
        'description': 'Standard setup with nominal noise levels',
        'param_noise': '15%',
        'obs_noise': '10%',
        'timepoints': 6
    },
    {
        'name': 'Sparse Data',
        'description': 'Reduced measurement timepoints',
        'param_noise': '15%',
        'obs_noise': '10%',
        'timepoints': 3
    },
    {
        'name': 'High Noise',
        'description': 'Increased observation noise',
        'param_noise': '15%',
        'obs_noise': '25%',
        'timepoints': 6
    },
    {
        'name': 'Misspecified ODE',
        'description': 'Model mismatch in known dynamics',
        'param_noise': '15%',
        'obs_noise': '10%',
        'timepoints': 6
    },
    {
        'name': 'Reduced Sampling',
        'description': 'Limited patient cohort',
        'param_noise': '15%',
        'obs_noise': '10%',
        'timepoints': 6
    },
    {
        'name': 'Multi-resolution',
        'description': 'Heterogeneous timepoint schedules',
        'param_noise': '15%',
        'obs_noise': '10%',
        'timepoints': 'variable'
    }
]
for i, exp in enumerate(experiments, 1):
    print(f'{i}. {exp["name"]}')
    print(f'   {exp["description"]}')
    print(f'   Param noise: {exp["param_noise"]}, Obs noise: {exp["obs_noise"]}, Timepoints: {exp["timepoints"]}')
    print()

Benchmark Experiments:

1. Baseline
   Standard setup with nominal noise levels
   Param noise: 15%, Obs noise: 10%, Timepoints: 6

2. Sparse Data
   Reduced measurement timepoints
   Param noise: 15%, Obs noise: 10%, Timepoints: 3

3. High Noise
   Increased observation noise
   Param noise: 15%, Obs noise: 25%, Timepoints: 6

4. Misspecified ODE
   Model mismatch in known dynamics
   Param noise: 15%, Obs noise: 10%, Timepoints: 6

5. Reduced Sampling
   Limited patient cohort
   Param noise: 15%, Obs noise: 10%, Timepoints: 6

6. Multi-resolution
   Heterogeneous timepoint schedules
   Param noise: 15%, Obs noise: 10%, Timepoints: variable



## Evaluation Metrics

How methods are compared and evaluated.

In [7]:
print('Evaluation Metrics:')
print('=' * 70)
print()
print('1. STATE VARIABLE RECOVERY')
print('   Metric: R-squared = 1 - (SS_residual / SS_total)')
print('   Computed for: P (pathogen), D (DAMP), h (tissue damage)')
print('   Scale: 0-1 (1 = perfect recovery)')
print()
print('2. OUTCOME CLASSIFICATION')
print('   Metric: AUROC (Area Under ROC Curve)')
print('   Task: Predict outcome (resolution/chronic/death)')
print('   One-vs-rest approach for multiclass')
print()
print('3. FEATURE IMPORTANCE')
print('   Metric: Correlation (Spearman) with outcome')
print('   Identifies biomarkers')
print()
print('4. COMPUTATIONAL EFFICIENCY')
print('   Metric: Wall-clock time per patient')
print('   MOTIF: ~1-5s (calibration + proxies)')
print('   UDE: ~100-500s (depends on epochs)')
print()
print('5. INTERPRETABILITY')
print('   MOTIF: Fitted parameter values are interpretable')
print('   UDE: Learned function requires SINDy for interpretation')

Evaluation Metrics:

1. STATE VARIABLE RECOVERY
   Metric: R-squared = 1 - (SS_residual / SS_total)
   Computed for: P (pathogen), D (DAMP), h (tissue damage)
   Scale: 0-1 (1 = perfect recovery)

2. OUTCOME CLASSIFICATION
   Metric: AUROC (Area Under ROC Curve)
   Task: Predict outcome (resolution/chronic/death)
   One-vs-rest approach for multiclass

3. FEATURE IMPORTANCE
   Metric: Correlation (Spearman) with outcome
   Identifies biomarkers

4. COMPUTATIONAL EFFICIENCY
   Metric: Wall-clock time per patient
   MOTIF: ~1-5s (calibration + proxies)
   UDE: ~100-500s (depends on epochs)

5. INTERPRETABILITY
   MOTIF: Fitted parameter values are interpretable
   UDE: Learned function requires SINDy for interpretation


## Reproducibility

Guide to reproducing the benchmark results.

In [8]:
print('Reproducibility Guide:')
print('=' * 70)
print()
print('1. SEEDING STRATEGY')
print('   - Master seed: Set before cohort generation')
print('   - Patient seed: master_seed + patient_id')
print('   - Ensures: Same P0 and parameters for same patient across runs')
print()
print('2. RUN EXPERIMENTS')
print('   Command line:')
print('   $ python src/run_experiment.py --config config_baseline.yaml')
print()
print('   Arguments:')
print('   --config:   Path to YAML config file (required)')
print('   --verbose:  Print progress messages (optional flag)')
print()
print('   Note: random_seed and output_dir are set in the YAML config file.')
print()
print('3. EXPECTED OUTPUTS')
print('   - results/{timestamp}/synthetic_cohort.pkl')
print('   - results/{timestamp}/motif_calibrations.pkl')
print('   - results/{timestamp}/ude_model.pt')
print('   - results/{timestamp}/metrics.json')
print()
print('4. HARDWARE CONSIDERATIONS')
print('   - GPU: Recommended for UDE training (much faster)')
print('   - CPU: ~30-60min for full benchmark')
print('   - RAM: ~2GB for 500 patients')
print()
print('5. DEPENDENCIES')
print('   - Core: numpy, scipy, scikit-learn, matplotlib')
print('   - ODE: torch, torchdiffeq')
print('   - Optional: pysindy (symbolic regression)')

Reproducibility Guide:

1. SEEDING STRATEGY
   - Master seed: Set before cohort generation
   - Patient seed: master_seed + patient_id
   - Ensures: Same P0 and parameters for same patient across runs

2. RUN EXPERIMENTS
   Command line:
   $ python src/run_experiment.py --config config_baseline.yaml

   Arguments:
   --config:   Path to YAML config file (required)
   --verbose:  Print progress messages (optional flag)

   Note: random_seed and output_dir are set in the YAML config file.

3. EXPECTED OUTPUTS
   - results/{timestamp}/synthetic_cohort.pkl
   - results/{timestamp}/motif_calibrations.pkl
   - results/{timestamp}/ude_model.pt
   - results/{timestamp}/metrics.json

4. HARDWARE CONSIDERATIONS
   - GPU: Recommended for UDE training (much faster)
   - CPU: ~30-60min for full benchmark
   - RAM: ~2GB for 500 patients

5. DEPENDENCIES
   - Core: numpy, scipy, scikit-learn, matplotlib
   - ODE: torch, torchdiffeq
   - Optional: pysindy (symbolic regression)


## Clinical Translation: From Computational Model to Patient Inference

![From Digital Twin to Clinical Decision Support](../figures/schematics/Fig%20C%20-%20ChatGPT%20Image%20Jul%2021%202026%2009_53_10%20PM%20-%20best.png)

## Mathematical Details

Key equations and derivations.

In [9]:
print('Reynolds ODE System:')
print('=' * 70)
print()
print('State variables:')
print('  P:     Pathogen load')
print('  N*:    Early pro-inflammatory mediator')
print('  D:     Late pro-inflammatory/DAMP mediator')
print('  CA:    Anti-inflammatory mediator (antibodies)')
print('  f:     Tissue damage fraction')
print('  h:     Tissue health = 1 - f')
print()
print('Core dynamics:')
print('  dP/dt   = k_pg*P*(1 - P/P_inf) - k_pm*N*/(s_dm + N*)*P - mu_p*P')
print('  dN*/dt  = s_nr/(1 + (CA/eps_nr)^2)*[P/(s_nm+P) + D/(s_nd+D)]*(1-N*/N_inf) - mu_nr*N*')
print('  dD/dt   = k_dn*N* + k_df*f - mu_d*D')
print('  dCA/dt  = s_c*N*^2/(s_cn^2 + N*^2) - mu_c*CA')
print('  df/dt   = k_f*N**(1-f) - k_fh*h*f')
print()
print('Key features:')
print('  - Negative feedback: CA suppresses N* production (epsilon_nr term)')
print('  - Pathogen-driven damage: N* drives both D and f')
print('  - Recovery mechanism: Tissue repair (k_fh) and pathogen clearance')

Reynolds ODE System:

State variables:
  P:     Pathogen load
  N*:    Early pro-inflammatory mediator
  D:     Late pro-inflammatory/DAMP mediator
  CA:    Anti-inflammatory mediator (antibodies)
  f:     Tissue damage fraction
  h:     Tissue health = 1 - f

Core dynamics:
  dP/dt   = k_pg*P*(1 - P/P_inf) - k_pm*N*/(s_dm + N*)*P - mu_p*P
  dN*/dt  = s_nr/(1 + (CA/eps_nr)^2)*[P/(s_nm+P) + D/(s_nd+D)]*(1-N*/N_inf) - mu_nr*N*
  dD/dt   = k_dn*N* + k_df*f - mu_d*D
  dCA/dt  = s_c*N*^2/(s_cn^2 + N*^2) - mu_c*CA
  df/dt   = k_f*N**(1-f) - k_fh*h*f

Key features:
  - Negative feedback: CA suppresses N* production (epsilon_nr term)
  - Pathogen-driven damage: N* drives both D and f
  - Recovery mechanism: Tissue repair (k_fh) and pathogen clearance


## References
- Reynolds et al. (2006). A reduced mathematical model of acute inflammatory response. J Theor Biol.
- Funk et al. (2025). MOTIF: Multi-Omics Temporal Integration Framework.
- Rackauckas & Nie (2020). Universal Differential Equations. arXiv.
- Brunton et al. (2016). Discovering governing equations from data. PNAS. For more information, see README.md and TECHNICAL_SPEC.md